In [ ]:
import pandas as pd

ruta = "DATA_ROOT/df_labevents_pivoted_1.xlsx"
df = pd.read_excel(ruta)
df.head()

In [ ]:
import pandas as pd
from sklearn.impute import KNNImputer


# Calcular porcentaje de valores nulos por columna
null_percentages = df.isnull().mean().sort_values(ascending=False) * 100

# Copia del DataFrame para el proceso de imputación
df_imputado = df.copy()

# 1. Eliminar columnas con más del 60% de nulos
columnas_1 = null_percentages[null_percentages > 60].index.tolist()
df_imputado.drop(columns=columnas_1, inplace=True)

# 2. Imputar columnas con 40-45% de nulos por mediana
columnas_2 = null_percentages[(null_percentages > 40) & (null_percentages <= 45)].index.tolist()
for col in columnas_2:
    if col in df_imputado.columns:
        df_imputado[col].fillna(df_imputado[col].median(), inplace=True)

# 3. Imputar columnas con 25-30% de nulos por mediana
columnas_3 = null_percentages[(null_percentages > 25) & (null_percentages <= 30)].index.tolist()
for col in columnas_3:
    if col in df_imputado.columns:
        df_imputado[col].fillna(df_imputado[col].median(), inplace=True)

# 4. Imputar columnas con <20% de nulos usando KNNImputer
columnas_4 = null_percentages[null_percentages < 20].index.tolist()
columnas_4 = [col for col in columnas_4 if col in df_imputado.columns and df_imputado[col].isnull().sum() > 0]
columnas_4 = [col for col in columnas_4 if pd.api.types.is_numeric_dtype(df_imputado[col])]

# Aplicar KNN Imputer si hay columnas válidas
if columnas_4:
    imputer = KNNImputer(n_neighbors=5)
    df_imputado[columnas_4] = imputer.fit_transform(df_imputado[columnas_4])



# Seleccionar las 4 columnas mencionadas y otras que pueden ayudar al modelo KNN
columnas_objetivo = [
    'Chloride-Blood-Chemistry',
    'Creatinine-Blood-Chemistry',
    'Potassium-Blood-Chemistry',
    'Sodium-Blood-Chemistry'
]

# Para que el KNN funcione bien, añadimos algunas variables clínicas relacionadas
columnas_contexto = [
    'Urea Nitrogen-Blood-Chemistry',
    'Bicarbonate-Blood-Chemistry',
    'Anion Gap-Blood-Chemistry',
    'Calcium, Total-Blood-Chemistry',
    'Magnesium-Blood-Chemistry'
]

# Filtrar las columnas que existan y sean numéricas
columnas_totales = [col for col in columnas_objetivo + columnas_contexto if col in df.columns and pd.api.types.is_numeric_dtype(df[col])]

# Aplicar KNNImputer solo sobre esas columnas
df_knn = df[columnas_totales].copy()
imputer = KNNImputer(n_neighbors=5)
df_knn_imputado = pd.DataFrame(imputer.fit_transform(df_knn), columns=df_knn.columns)

# Sustituir las columnas objetivo imputadas en el DataFrame original
for col in columnas_objetivo:
    if col in df_knn_imputado.columns:
        df[col] = df_knn_imputado[col]
    
    
    

# Guardar el nuevo archivo imputado
df_imputado.to_excel("df_labevents_imputado_custom_vf.xlsx", index=False)


- Con el objetivo de garantizar la calidad y fiabilidad del análisis de datos, se aplicó una estrategia combinada para el tratamiento de valores ausentes en las variables de laboratorio de pacientes con diabetes. Dicha estrategia tuvo en cuenta el porcentaje de datos faltantes, la relevancia clínica de cada biomarcador y la coherencia entre variables fisiológicamente relacionadas.

- En primer lugar, se eliminaron aquellas variables con más del 60% de datos ausentes, por considerarse poco representativas y con escasa utilidad analítica. Posteriormente, las variables con entre 25% y 45% de nulos fueron imputadas utilizando la mediana de cada variable, al tratarse de una medida robusta frente a posibles valores atípicos y adecuada para distribuciones no simétricas.

- Para aquellas variables con menos del 20% de datos ausentes, se optó por una imputación basada en K-Nearest Neighbors (KNN), utilizando K=5. Este algoritmo permite estimar los valores faltantes en función de la similitud clínica entre pacientes, considerando un conjunto de variables relevantes (como urea, anión gap, bicarbonato, etc.) para generar imputaciones más precisas y contextualizadas.

Adicionalmente, se aplicó una imputación específica por KNN para un subconjunto de variables críticas en el estudio:

**Chloride-Blood-Chemistry
** Creatinine-Blood-Chemistry
** Potassium-Blood-Chemistry
** Sodium-Blood-Chemistry**

- Estas variables, estrechamente ligadas al equilibrio ácido-base y la función renal, fueron imputadas con especial cuidado utilizando otras variables bioquímicas relacionadas como predictores auxiliares. Este tratamiento se realizó incluso si superaban el umbral del 20% de nulos, dada su alta relevancia clínica para el desarrollo de modelos predictivos en pacientes con diabetes.

- Esta estrategia híbrida permitió conservar la mayor cantidad de información sin comprometer la integridad analítica del conjunto de datos, alineándose con buenas prácticas en el manejo de datos clínicos y asegurando imputaciones coherentes con el perfil fisiopatológico de los pacientes.

   ## HISTOGRAMA DE FRECUENCIAS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar el archivo subido

file_path = "DATA_ROOT/df_labevents_imputado_custom_vf.xlsx"
df = pd.read_excel(file_path)

# Seleccionar columnas numéricas para graficar
columnas_numericas = df.select_dtypes(include='number').columns

# Crear histogramas para cada columna numérica
for col in columnas_numericas:
    plt.figure(figsize=(6, 4))
    df[col].hist(bins=30)
    plt.title(f'Distribución de {col}')
    plt.xlabel('Valor')
    plt.ylabel('Frecuencia')
    plt.grid(True)
    plt.tight_layout()
    plt.show()


- Se generaron histogramas para todas las variables numéricas incluidas en el dataset de laboratorio tras la imputación, con el objetivo de analizar su distribución, detectar posibles valores atípicos y evaluar la consistencia clínica de los datos.

- La mayoría de los biomarcadores mostraron distribuciones asimétricas, con una gran concentración de valores dentro de los rangos fisiológicos esperados y una cola derecha más extendida, indicativa de la presencia de pacientes en estados clínicos alterados o con mediciones extremas.

Entre los hallazgos más destacados:

- **Creatinine-Blood-Chemistry** presentó una distribución sesgada a la derecha, con un pico marcado en torno a los 100–150 µmol/L, coherente con valores normales, pero con una dispersión considerable hacia valores superiores, lo cual es esperable en una población con disfunción renal como los pacientes diabéticos.

- **Potassium-Blood-Chemistry y Sodium-Blood-Chemistry** mostraron curvas bien definidas, centradas en sus rangos normales clínicos (3.5–5.5 mEq/L para potasio y 135–145 mEq/L para sodio), aunque con algunos valores extremos que podrían corresponder a errores de medición o a estados clínicos críticos (hiper o hiponatremia).

- **Chloride-Blood-Chemistry** mostró una distribución anómala, con la presencia de valores extremadamente elevados (por encima de 1000), lo que sugiere la existencia de errores de codificación, escala o unidad en la base de datos original. Este patrón fue claramente visible en el histograma y requiere ser corregido o depurado antes de utilizar la variable en análisis posteriores.

- Otras variables como **Hemoglobin, Platelets, RDW o Bicarbonate** también revelaron patrones coherentes con lo esperado clínicamente, aunque algunas mostraron colas alargadas hacia valores elevados, lo cual podría estar reflejando tanto la variabilidad entre pacientes como la presencia de episodios agudos o crónicos en el curso de la enfermedad.

- Este análisis visual fue clave para complementar la imputación previa y permitió identificar variables que, a pesar de haber sido tratadas estadísticamente, aún requerían validación clínica adicional o transformación para su análisis en modelos predictivos.